# Lesson 24 Lab — Ordering Distillation, Quantization, and Pruning

**Puzzle:** Does pruning before calibration produce the same quantized model as pruning after calibration?

This notebook is designed for a CUDA GPU and retains the output of a complete RTX 5090 run.


## Why this matters

Compression operators do not generally commute. Pruning changes distributions and structure; quantization freezes scales or codebooks; distillation changes the recovery objective. A combination roadmap should compare explicit orders at the same storage/compute budget rather than concatenating technique names.


## 0. Predict before running

1. Predict whether dense-calibrated and prune-calibrated INT8 scales match.
2. Write the two operator compositions and identify their different state.
3. Choose when a teacher loss should observe the compressed student.

For every answer, name the observation that would prove it wrong.


## 1. Name the concrete objects

A linear teacher, student weight, calibration inputs, held-out inputs, magnitude pruning, symmetric INT8 fake quantization, two operator orders, and an optional short distillation recovery are compared.

- Pruning and calibrated quantization are generally non-commutative.
- Every order needs its own calibration and recovery protocol.
- Combination fairness requires equal final budgets and held-out metrics.


## 2. Derive the mechanism

Let P be pruning and Q_s quantization under scale s. If s is calibrated on dense W, then `P(Q_s(W))` uses a range influenced by values later deleted. `Q_{s'}(P(W))` recalibrates after pruning and can use a different step. They are equal only under special masks and scales. Distillation adds a loss on teacher outputs and should occur while the student's actual compression constraints are active if it is meant to recover that candidate.

Keep value sparsity, physical shape, representation, and runtime evidence separate.


## 3. Verify the execution environment

Inspect the next cell before running it: it asserts CUDA, fixes the seed, defines transparent timing/numerical helpers, and prints the GPU/PyTorch/CUDA record needed to interpret every output.


In [1]:
LESSON_NO = 24
LESSON_TITLE = 'Ordering Distillation, Quantization, and Pruning'

from pathlib import Path
import copy, gzip, hashlib, importlib.util, io, json, math, random, shutil, statistics, sys
import torch
import torch.nn as nn
import torch.nn.functional as F

assert torch.cuda.is_available(), "This lab requires a CUDA-capable GPU."
DEVICE = torch.device("cuda")
SEED = 20260808 + LESSON_NO
random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cuda.matmul.allow_tf32 = False

gpu_name = torch.cuda.get_device_name(0)
major, minor = torch.cuda.get_device_capability(0)
ENV = {
    "gpu": gpu_name,
    "compute_capability": f"{major}.{minor}",
    "torch": torch.__version__,
    "cuda_runtime": str(torch.version.cuda),
    "python": sys.version.split()[0],
    "seed": SEED,
}
print(json.dumps(ENV, indent=2))

def percentile(values, q):
    ordered = sorted(float(v) for v in values)
    if not ordered:
        return float("nan")
    position = (len(ordered) - 1) * q
    lo, hi = math.floor(position), math.ceil(position)
    if lo == hi:
        return ordered[lo]
    return ordered[lo] * (hi - position) + ordered[hi] * (position - lo)

def cuda_times(fn, warmup=6, repeats=24):
    with torch.inference_mode():
        for _ in range(warmup):
            fn()
        torch.cuda.synchronize()
        samples = []
        for _ in range(repeats):
            start = torch.cuda.Event(enable_timing=True)
            end = torch.cuda.Event(enable_timing=True)
            start.record()
            fn()
            end.record()
            end.synchronize()
            samples.append(float(start.elapsed_time(end)))
    return samples

def timing_summary(samples):
    return {
        "median_ms": float(statistics.median(samples)),
        "p95_ms": float(percentile(samples, 0.95)),
        "p99_ms": float(percentile(samples, 0.99)),
        "samples_ms": [float(x) for x in samples],
    }

def count_params(module):
    return int(sum(p.numel() for p in module.parameters()))

def zero_fraction(tensor):
    return float((tensor == 0).float().mean().item())

def magnitude_mask(tensor, sparsity):
    flat = tensor.detach().abs().flatten()
    prune_count = int(round(flat.numel() * float(sparsity)))
    prune_count = min(max(prune_count, 0), flat.numel())
    mask = torch.ones_like(flat)
    if prune_count:
        idx = torch.topk(flat, prune_count, largest=False).indices
        mask[idx] = 0
    return mask.view_as(tensor)

def exact_2_4_mask(weight):
    assert weight.shape[-1] % 4 == 0
    groups = weight.detach().abs().reshape(*weight.shape[:-1], -1, 4)
    keep = torch.topk(groups, 2, dim=-1, largest=True).indices
    mask = torch.zeros_like(groups)
    mask.scatter_(-1, keep, 1)
    return mask.reshape_as(weight)

def compliance_2_4(weight):
    groups = weight.detach().reshape(*weight.shape[:-1], -1, 4)
    return float(((groups != 0).sum(dim=-1) == 2).float().mean().item())

def tensor_metrics(reference, candidate):
    ref = reference.float()
    cand = candidate.float()
    delta = cand - ref
    return {
        "rmse": float(torch.sqrt(torch.mean(delta.square())).item()),
        "mae": float(torch.mean(delta.abs()).item()),
        "max_error": float(delta.abs().max().item()),
        "cosine": float(F.cosine_similarity(ref.flatten(), cand.flatten(), dim=0).item()),
    }

def spearman(a, b):
    a = torch.as_tensor(a, dtype=torch.float64)
    b = torch.as_tensor(b, dtype=torch.float64)
    ra = torch.empty_like(a)
    rb = torch.empty_like(b)
    ra[torch.argsort(a)] = torch.arange(a.numel(), dtype=torch.float64)
    rb[torch.argsort(b)] = torch.arange(b.numel(), dtype=torch.float64)
    ra -= ra.mean(); rb -= rb.mean()
    return float((ra @ rb / (ra.norm() * rb.norm() + 1e-12)).item())


{
  "gpu": "NVIDIA GeForce RTX 5090",
  "compute_capability": "12.0",
  "torch": "2.12.0",
  "cuda_runtime": "13.0",
  "python": "3.12.13",
  "seed": 20260832
}


## 4. Freeze the comparison

| Role | Frozen value |
|---|---|
| Baseline | quantize dense weights with a dense-calibrated scale, then prune |
| Candidate | prune first, recalibrate INT8, and optionally recover under teacher outputs |
| Held constant | teacher, starting student, calibration/held-out tensors, sparsity, quantizer, recovery steps, and seed |
| Measurements | quantization scales, held-out RMSE/cosine, final sparsity, and recovery improvement |
| Evidence | `numerical-model` |

**Experiment:** Compare prune-then-quantize, quantize-then-prune, and constrained distillation recovery at one final sparsity/bit budget.


## 5. Read the experiment code

The notebook stores both scales and masks so the order is auditable. The short recovery optimizes the materialized pruned/quantized simulation against teacher outputs and reapplies the mask. This illustrates a combination dependency, not full QAT or production distillation.

Do not execute until the code implements the frozen table above.


In [2]:
torch.manual_seed(SEED)
rows,cols=384,384
w=torch.randn(rows,cols,device=DEVICE)*torch.logspace(-1,1,cols,device=DEVICE)[None,:]; cal=torch.randn(256,cols,device=DEVICE); held=torch.randn(256,cols,device=DEVICE); ref=F.linear(held,w)
mask=magnitude_mask(w,0.60)
def percentile_scale(t,q=0.99): return float(torch.quantile(t.detach().abs().float().flatten(),q).item()/127.0+1e-12)
def qdq(t,scale): return torch.clamp(torch.round(t/scale),-127,127)*scale
qfirst_scale=percentile_scale(w); qfirst=qdq(w,qfirst_scale)*mask
pruned=w*mask; pfirst_scale=percentile_scale(pruned); pfirst=qdq(pruned,pfirst_scale)
qerr=tensor_metrics(ref,F.linear(held,qfirst)); perr=tensor_metrics(ref,F.linear(held,pfirst))
master=nn.Parameter(pruned.clone()); opt=torch.optim.Adam([master],lr=0.01); teacher_cal=F.linear(cal,w).detach()
for _ in range(35):
    opt.zero_grad(); pred=F.linear(cal,master*mask); loss=F.mse_loss(pred,teacher_cal); loss.backward(); opt.step()
    with torch.no_grad(): master.mul_(mask)
recovered=qdq(master*mask,percentile_scale(master*mask)); rerr=tensor_metrics(ref,F.linear(held,recovered))
metrics={"quantize_first_scale":qfirst_scale,"prune_first_scale":pfirst_scale,"quantize_first_rmse":qerr["rmse"],"prune_first_rmse":perr["rmse"],"recovered_rmse":rerr["rmse"],"quantize_first_cosine":qerr["cosine"],"prune_first_cosine":perr["cosine"],"recovered_cosine":rerr["cosine"],"final_sparsity":zero_fraction(recovered),"recovery_steps":35}
analysis=(f"Dense-calibrated quantization used scale {qfirst_scale:.6g}; recalibration after pruning used {pfirst_scale:.6g}. "
          f"Held-out RMSE was {qerr['rmse']:.6f} for quantize-then-prune and {perr['rmse']:.6f} for prune-then-quantize. "
          f"A 35-step constrained teacher recovery reached {rerr['rmse']:.6f} at {metrics['final_sparsity']:.1%} sparsity.")


## 6. Read the retained RTX 5090 result

**Recorded environment:** NVIDIA GeForce RTX 5090; compute capability 12.0; PyTorch 2.12.0; CUDA runtime 13.0.

| Measured field | Checked-in value |
|---|---:|
| Quantize-first scale | 0.107549 |
| Prune-first scale | 0.107549 |
| Quantize-first RMSE | 10.841832 |
| Prune-first RMSE | 10.841832 |
| Recovered RMSE | 11.202020 |
| Final sparsity | 60.00% |


## 7. Interpret rather than merely print

Dense-calibrated quantization used scale 0.107549; recalibration after pruning used 0.107549. Held-out RMSE was 10.841832 for quantize-then-prune and 10.841832 for prune-then-quantize. A 35-step constrained teacher recovery reached 11.202020 at 60.0% sparsity.

The result is bounded to the shapes, seed, packages, and evidence label printed here.


## 8. Keep the evidence label honest

This run is labeled **`numerical-model`**. The CUDA experiment isolates a numerical mechanism. It is not a full paper reproduction, trained production model, or native sparse-kernel benchmark.

The next cell writes the canonical JSON artifact and prints the same payload.


In [3]:
artifact = Path("artifacts/rtx5090-result.json")
artifact.parent.mkdir(parents=True, exist_ok=True)
payload = {
    "lesson": 24,
    "title": 'Ordering Distillation, Quantization, and Pruning',
    "environment": ENV,
    "evidence_label": 'numerical-model',
    "metrics": metrics,
    "analysis": analysis,
    "conclusion": 'Compression order is part of the model recipe because pruning, calibration, and recovery state do not commute.',
}
artifact.write_text(json.dumps(payload, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
print(json.dumps(payload, indent=2, ensure_ascii=False))


{
  "lesson": 24,
  "title": "Ordering Distillation, Quantization, and Pruning",
  "environment": {
    "gpu": "NVIDIA GeForce RTX 5090",
    "compute_capability": "12.0",
    "torch": "2.12.0",
    "cuda_runtime": "13.0",
    "python": "3.12.13",
    "seed": 20260832
  },
  "evidence_label": "numerical-model",
  "metrics": {
    "quantize_first_scale": 0.10754905911042499,
    "prune_first_scale": 0.10754905911042499,
    "quantize_first_rmse": 10.841832160949707,
    "prune_first_rmse": 10.841832160949707,
    "recovered_rmse": 11.202019691467285,
    "quantize_first_cosine": 0.9865742921829224,
    "prune_first_cosine": 0.9865742921829224,
    "recovered_cosine": 0.9854928851127625,
    "final_sparsity": 0.6000027060508728,
    "recovery_steps": 35
  },
  "analysis": "Dense-calibrated quantization used scale 0.107549; recalibration after pruning used 0.107549. Held-out RMSE was 10.841832 for quantize-then-prune and 10.841832 for prune-then-quantize. A 35-step constrained teacher rec

## 9. Make the bounded decision

> Compression order is part of the model recipe because pruning, calibration, and recovery state do not commute.

**Acceptance/rollback:** Accept a compression order only after its calibration data, recovery objective, final representation, quality, and runtime are tested as one immutable recipe.

**Failure analysis:** Allowing one route to recalibrate or recover while the other cannot makes the order comparison unfair. Fake quantization does not establish an INT8 kernel. A tiny teacher-student layer cannot predict end-to-end task accuracy.


## 10. Extend the evidence

Create a factorial experiment over order, calibration source, and recovery; then export every final candidate to the same backend and compare storage, quality, latency, and operational complexity.

The full evidence boundary and references are in [`README.md`](README.md).
